This is our notebook

Here, we load the dataset in using the pandas built in library for reading through csv files.

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv("training_v2.csv")
df.head()


,encounter_id,patient_id,hospital_id,hospital_death,age,bmi,elective_surgery,ethnicity,gender,height,...,aids,cirrhosis,diabetes_mellitus,hepatic_failure,immunosuppression,leukemia,lymphoma,solid_tumor_with_metastasis,apache_3j_bodysystem,apache_2_bodysystem
0,66154,25312,118,0,68.0,22.73,0,Caucasian,M,180.3,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Sepsis,Cardiovascular
1,114252,59342,81,0,77.0,27.42,0,Caucasian,F,160.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Respiratory,Respiratory
2,119783,50777,118,0,25.0,31.95,0,Caucasian,F,172.7,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Metabolic,Metabolic
3,79267,46918,118,0,81.0,22.64,1,Caucasian,F,165.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Cardiovascular,Cardiovascular
4,92056,34377,33,0,19.0,NaN,0,Caucasian,M,188.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Trauma,Trauma


Unlabeled dataset (dataset.csv)

This is typically:
Use this after training to: Make predictions, Simulate “real-world” unseen data, this will be our analysis. 

clean data, encode categorical variables and define features and target

In [16]:
# Drop ID-like columns (not useful for prediction)
df = df.drop(columns=["encounter_id", "patient_id"], errors="ignore")

# Handle missing values
# numeric fill
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# categorical fill
cat_cols = df.select_dtypes(exclude=["number"]).columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

df = pd.get_dummies(df, drop_first=True)

X = df.drop("hospital_death", axis=1)
y = df["hospital_death"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_sub = X_train.sample(n = 8000, random_state = 42)
y_train_sub = y_train.loc[X_train_sub.index]

baselime nodel will be a logistic regression because the model has to be basic not the data apparently...

Logistic regression was used as a linear probabilistic classifier representing a simple decision boundary.\

Logistic regression convergence depends on feature scaling and solver choice. High-dimensional sparse feature spaces can slow optimization and require more robust solvers such as SAGA.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

#baseline_model = LogisticRegression(max_iter=1000)
baseline_model = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter = 2000, solver = 'lbfgs'))])

baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

the actual model basic version, im also going to do at least one improved version where we can try with more dataparsing or specific settings

A support vector machine with an RBF kernel was used to construct a nonlinear maximum-margin classifier.

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

basic_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True))
])

basic_svm.fit(X_train_sub, y_train_sub)

basic_svm_preds = basic_svm.predict(X_test)
basic_svm_probs = basic_svm.predict_proba(X_test)[:, 1]

adding hyper parameters

The effect of hyperparameters (C and gamma) was evaluated to demonstrate changes in model complexity and overfitting behavior.

In [19]:
svm_tuned = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma=0.01, probability=True))
])

svm_tuned.fit(X_train_sub, y_train_sub)

svm_tuned_preds = svm_tuned.predict(X_test)
svm_tuned_probs = svm_tuned.predict_proba(X_test)[:, 1]

adding another svm that has less feature but still not hyperparamenters, Reducing the feature space allowed us to analyze the effect of dimensionality on SVM performance

In [20]:
selected_features = [
    "age",
    "bmi",
    "heart_rate_apache",
    "gcs_motor_apache",
    "d1_bun_max",
    "d1_creatinine_max"
]

X_train_reduced = X_train_sub[selected_features]
X_test_reduced = X_test[selected_features]

svm_reduced = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1, gamma="scale", probability=True))
])

svm_reduced.fit(X_train_reduced, y_train_sub)

svm_reduced_preds = svm_reduced.predict(X_test_reduced)
svm_reduced_probs = svm_reduced.predict_proba(X_test_reduced)[:, 1]

In [ ]:

models = {"logisic regression (baseline)": baseline_probs, "basic svm" : basic_svm_probs, "tuned svm": svm_tuned_probs, "reduced feature svm": svm_reduced_probs}

for name, probs in models.items():
    auc = roc_auc_score(y_test, probs)
    print(f'{name} - ROC-AUC Score: {auc:.4f}')

logisic regression (baseline) - ROC-AUC Score: 0.8832
basic svm - ROC-AUC Score: 0.8525
tuned svm - ROC-AUC Score: 0.8416
reduced feature svm - ROC-AUC Score: 0.6200


In [24]:
unlabeled_df = pd.read_csv("unlabeled.csv")

submission_ids = unlabeled_df['encounter_id']

unlabeled_df = unlabeled_df.drop(columns=["encounter_id", "patient_id"], errors="ignore")
train_medians = df[num_cols].median()
unlabeled_df[num_cols] = unlabeled_df[num_cols].fillna(train_medians)

unlabeled_df[cat_cols] = unlabeled_df[cat_cols].fillna("Unknown")
X_unlabeled = pd.get_dummies(unlabeled_df, drop_first = True)

X_unlabeled = X_unlabeled.reindex(columns = X.columns, fill_value = 0)

final_probs = baseline_model.predict_proba(X_unlabeled)[:, 1]

submission = pd.read_csv("solution_template.csv")

submission["hospital_death"] = final_probs
submission.to_csv("my_final_predictions.csv", index = False)

fin = pd.read_csv("my_final_predictions.csv")
fin.head()

,encounter_id,hospital_death
0,2,0.044505
1,5,0.010659
2,7,0.018282
3,8,0.104474
4,10,0.245887
